# Trailwise · Presentation walkthrough
Choose **Run all**. On a hosted Colab CPU, the notebook loads the reviewed presentation release, runs a lightweight retrieval preview and displays the frozen evaluation and three-turn demonstration. On a configured Apple Silicon Mac, the last cell also starts or reuses the same web app as `run-abc-live.command` and requests two fresh answers.

In [ ]:
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

# Use the local project when available. Colab opens the notebook by itself.
notebook_path = Path("notebooks/abc_rag_walkthrough.ipynb")
root = next(((p / notebook_path).resolve().parents[1]
             for p in (Path.cwd(), *Path.cwd().parents)
             if (p / notebook_path).is_file() and (p / "app/config.py").is_file()), None)
if root is None:
    if "download_folder" not in globals():
        download_folder = TemporaryDirectory(prefix="trailwise-")
    root = Path(download_folder.name) / "trailwise-abc"
    if not (root / ".git").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", "main",
                        "https://github.com/Chen-Xingjian/trailwise-abc.git", str(root)], check=True)

manifest_path = Path("documents/abc-six-day/manifest.json")
if hashlib.sha256((root / manifest_path).read_bytes()).hexdigest() != "a861181bcb470910e15bd66975ba89fff4e3d6c088ca7428457b25b30f07a3f0":
    raise RuntimeError("The data release differs from the presentation. Reopen the matching notebook and checkout.")

# Install only the packages used for CPU retrieval and prompt construction.
packages = {"qdrant_client": "qdrant-client>=1.13,<2", "numpy": "numpy>=1.26,<3",
            "httpx": "httpx>=0.27,<1", "dotenv": "python-dotenv>=1,<2"}
if any(importlib.util.find_spec(module) is None for module in packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages.values()],
                   check=True)
    importlib.invalidate_caches()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Ready to load:", root / manifest_path)

## 1 · Validate the documents
The same manifest governs the ABC web launcher and this notebook. The loader checks file hashes, source rights reviews, accepted IDs and dependencies before constructing chunks.

In [ ]:
from app.config import Settings, application_revision
from app.release import load_release
from app.service import Assistant

corpus = root / manifest_path.parent
release = load_release(corpus)
revision = application_revision()
print("Presentation release:", release.manifest["release_id"])
print(f"{len(release.facts)} accepted facts, {len(release.sources)} source records, {len(release.chunks)} chunks")
print("Application fingerprint:", revision[:12])

## 2 · Inspect the uphill stage in the presentation
KeiTreks recorded both time moving and time including stops for the same Day 4 stage from Deurali to ABC. Each field has its own source locator and qualification.

In [ ]:
by_id = {chunk["fact_id"]: chunk for chunk in release.chunks}
for fact_id in ("KT-D4-M", "KT-D4-E"):
    example = by_id[fact_id]
    print(example["text"])
    for source in example["source_records"]:
        print(source["source_title"], source["source_locator"], source["licence_id"], sep=" | ")
    print()

## 3 · Run the CPU retrieval preview
Search the current release using the second question from the presentation. This preview uses word hashing and shows the prompt. The Mac web app uses the pinned Qwen embedding model and its full answer logic.

In [ ]:
demo = json.loads((root / "outputs/weather-stage-gear-first-answers.json").read_text())
question = demo[1]["question"]
with TemporaryDirectory() as temp:
    preview = Assistant(Settings(release_dir=corpus, runtime_dir=Path(temp), embedding_mode='hash'))
    try:
        hits = preview.retriever.search(question, limit=3)
        messages = preview.messages(question, hits)
    finally:
        preview.close()

print('Preview retrieval:', [hit['fact_id'] for hit in hits])
print(messages[0]['content'])
print(messages[1]['content'])

## 4 · Inspect the presentation results
The frozen offline comparison covers 16 scenarios. Each final answer contributes up to two points. The live three-turn weather, stage time and equipment demonstration was recorded separately. The output below replays those original answers, including the failed gear follow-up.

In [ ]:
scores = json.loads((root / "outputs/abc-control-blind-20261125/results.json").read_text())
final_turns = [row for row in scores if row["primary"]]
for label, rows in (("All final answers", final_turns),
                    ("Combined final turns", [r for r in final_turns if r["domain"] == "Combined"])):
    maximum = 2 * len(rows)
    rag = sum(row["rag"] for row in rows)
    baseline = sum(row["baseline"] for row in rows)
    print(f"{label}: with RAG {rag}/{maximum}, without RAG {baseline}/{maximum}")

for index, turn in enumerate(demo, 1):
    print(f"\nRecorded turn {index}\n{turn['displayed']}")

## 5 · Open the presentation app on Mac
On a configured Apple Silicon Mac, this cell calls the same launcher as the shell script. Both open the chat at the same address. The launcher checks the application, release, index and model settings before opening the page. The cell then asks the live app for a new pair of answers. If this kernel started the service, you can stop it with `presentation_process.terminate()`. Hosted Colab ends with the recorded demonstration.

In [ ]:
import platform
import time
from tempfile import NamedTemporaryFile
from urllib.request import ProxyHandler, Request, build_opener

mac_python = Path.home() / "Library/Caches/Trailwise/nepal-trail-rag/fresh-venv/bin/python"
if sys.platform == "darwin" and platform.machine() == "arm64" and mac_python.is_file():
    from scripts.launch_local import live_status

    base = "http://127.0.0.1:8765"
    opener = build_opener(ProxyHandler({}))
    launcher = [str(root / "run-abc-live.command")]
    env = {**os.environ, "DATA_RELEASE_DIR": str(corpus)}
    if live_status(opener) is not None:
        subprocess.run(launcher, cwd=root, env=env, check=True)
    else:
        if "presentation_process" not in globals() or presentation_process.poll() is not None:
            with NamedTemporaryFile(prefix="trailwise-launch-", suffix=".log", delete=False) as log:
                launch_log = Path(log.name)
                presentation_process = subprocess.Popen(launcher, cwd=root, env=env,
                                                        stdout=log, stderr=subprocess.STDOUT)
        for _ in range(480):
            log_text = launch_log.read_text(errors="replace")
            if "Trailwise is ready: " in log_text:
                break
            if presentation_process.poll() is not None:
                raise RuntimeError(log_text[-2000:] or "The local launcher exited before startup.")
            time.sleep(1)
        else:
            raise RuntimeError(f"Startup is still pending. See {launch_log}")
    status = live_status(opener)
    if not status or status.get("application_revision") != revision:
        raise RuntimeError("The local application differs from this notebook. Restart the kernel and launcher.")
    print("Live chat:", base, "|", status["release_id"], "|", status["indexed_fact_count"], "facts")
    for mode in ("rag", "baseline"):
        payload = {"question": question, "mode": mode, "top_k": 3, "offer_live_checks": False}
        request = Request(base + "/api/ask", data=json.dumps(payload).encode(),
                          headers={"Content-Type": "application/json"})
        with opener.open(request, timeout=180) as response:
            result = json.load(response)
        print(mode, result["answer_kind"], [hit["fact_id"] for hit in result["evidence"]])
        print(result["answer"])
else:
    print("CPU walkthrough complete. The three-turn conversation above is a recorded demonstration.")
    print("For the live chat, use the configured Apple Silicon Mac described in the README.")